In [1]:
"""
Per-layer requantization constants: M decomposition (ABACUS-7).

Each PTQ layer produces an int32 accumulator (int8 x int8 matmul + int32
bias). To feed the next layer we currently rescale by a float multiply
M = s_w * s_x_in / s_x_out. Real integer-only hardware has no float
multiplier in that path, so M is decomposed into a fixed-point int32
multiplier M0 and a right-shift n such that M ~= M0 * 2^-n, computed once
offline and applied on-device as one integer multiply + rounding shift.

Loads model/quant_params_ptq.npz (the PTQ path chosen for deployment) --
does not retrain or recalibrate anything.
"""

import math
import numpy as np
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

In [2]:
# ----------------------------------------------------------------------
# Load PTQ parameters
# ----------------------------------------------------------------------
PARAMS_PATH = "quant_params_ptq.npz"
p = np.load(PARAMS_PATH)

weight_int8 = {name: p[f"{name}_weight_int8"] for name in ("fc1", "fc2", "fc3")}
bias_int32  = {name: p[f"{name}_bias_int32"]  for name in ("fc1", "fc2", "fc3")}
s_w         = {name: float(p[f"{name}_s_w"])  for name in ("fc1", "fc2", "fc3")}
s_x         = {name: float(p[f"{name}_s_x"])  for name in ("fc1", "fc2", "fc3")}

ref_fp32_acc = float(p["fp32_test_acc"])
ref_int8_acc = float(p["int8_test_acc"])
print(f"loaded {PARAMS_PATH}, fp32 acc {ref_fp32_acc*100:.2f}%, PTQ int8 acc {ref_int8_acc*100:.2f}%")

loaded quant_params_ptq.npz, fp32 acc 89.51%, PTQ int8 acc 89.43%


In [3]:
# ----------------------------------------------------------------------
# M decomposition -- chosen scheme (write-up per ABACUS-7):
#
# M = significand * 2**exponent   (via math.frexp, significand in [0.5, 1))
# M0 = round(significand * 2**31)  -- unsigned Q0.31 fixed-point mantissa,
#      i.e. M0 is an int32 with 0 < M0 < 2**31, representing M0 / 2**31.
# shift = exponent                 -- signed; M ~= M0 * 2**(shift - 31).
#
# On-device: given an int32 accumulator `acc`,
#   total_shift = 31 - shift               (right-shift amount, >0 here)
#   rounded     = (acc * M0 + 2**(total_shift-1)) >> total_shift
# implements round-half-up on acc * M, using one N x 32-bit multiply into
# a >=48-bit-wide accumulator (acc * M0 can exceed int32 -- e.g. a ~24-bit
# accumulator times a 31-bit M0 needs up to ~55 bits, so the multiply-shift
# stage needs a wider register than the int8 MAC accumulator itself; a
# DSP48E2's 48-bit output register covers this for our accumulator sizes)
# followed by a right shift and truncation back to int8 with clamping.
#
# This is the standard gemmlowp/TFLite "quantized multiplier" scheme,
# chosen because it needs no division on-device (only a multiply, an add
# for rounding, and a shift) and the offline math is exact up to the
# 2**-31 resolution of M0.
# ----------------------------------------------------------------------
def quantize_multiplier(M):
    assert M > 0, "M decomposition assumes a positive rescale factor"
    significand, exponent = math.frexp(M)      # M = significand * 2**exponent
    M0 = round(significand * (1 << 31))
    if M0 == (1 << 31):                          # rounding pushed us to 1.0 exactly
        M0 //= 2
        exponent += 1
    assert 0 < M0 < (1 << 31)
    return M0, exponent


def apply_quantized_multiplier(acc, M0, shift):
    """acc: int64 tensor. Returns round(acc * M0 * 2**(shift-31)) as int64."""
    total_shift = 31 - shift
    acc64 = acc.to(torch.int64)
    if total_shift > 0:
        rounding = 1 << (total_shift - 1)
        return (acc64 * M0 + rounding) >> total_shift
    else:
        return acc64 * M0 << (-total_shift)

In [4]:
# ----------------------------------------------------------------------
# Derive M per transition. fc1 and fc2 rescale into the NEXT layer's
# input scale (s_y = s_x of the following layer). fc3 is the output
# layer -- argmax over a single positive per-tensor-scaled int32
# accumulator is scale-invariant, so no requantization (and no M) is
# needed there; we keep fc3's raw int32 accumulator for classification.
# ----------------------------------------------------------------------
TRANSITIONS = [("fc1", "fc2"), ("fc2", "fc3")]

M_const = {}
for src, dst in TRANSITIONS:
    M = s_w[src] * s_x[src] / s_x[dst]
    M0, shift = quantize_multiplier(M)
    M_const[src] = (M0, shift)
    M_approx = M0 * (2.0 ** (shift - 31))
    rel_err = abs(M_approx - M) / M
    print(f"{src}->{dst}: M={M:.8e}  M0={M0} ({M0:#010x})  shift={shift}  "
          f"total_right_shift={31-shift}  M0*2^(shift-31)={M_approx:.8e}  rel_err={rel_err:.3e}")

fc1->fc2: M=1.35986491e-03  M0=1495187284 (0x591ebf54)  shift=-9  total_right_shift=40  M0*2^(shift-31)=1.35986491e-03  rel_err=4.498e-11
fc2->fc3: M=3.84751576e-03  M0=2115194159 (0x7e134d2f)  shift=-8  total_right_shift=39  M0*2^(shift-31)=3.84751576e-03  rel_err=1.166e-10


In [5]:
# ----------------------------------------------------------------------
# Validate: run the full test set through a purely integer pipeline --
# int8 x int8 matmul, int32 bias add, integer multiply-by-M0 + rounding
# shift for requantization, ReLU as an int8 clamp-at-zero. Compare
# against the float-dequant PTQ reference and the fp32 baseline.
# ----------------------------------------------------------------------
MEAN, STD = 0.2860, 0.3530
tfm = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((MEAN,), (STD,)),
    transforms.Lambda(lambda x: x.view(-1)),
])
test_set = datasets.FashionMNIST("./data", train=False, download=True, transform=tfm)
test_loader = DataLoader(test_set, batch_size=512, shuffle=False, num_workers=0)

W = {name: torch.from_numpy(weight_int8[name]).to(torch.int64) for name in ("fc1", "fc2", "fc3")}
B = {name: torch.from_numpy(bias_int32[name]).to(torch.int64)  for name in ("fc1", "fc2", "fc3")}


@torch.no_grad()
def integer_only_forward(x_float):
    # Only place a float appears: quantizing the raw sensor/pixel input.
    # Everything past this line is integer add/multiply/shift.
    q = torch.clamp(torch.round(x_float / s_x["fc1"]), -127, 127).to(torch.int64)

    acc1 = q @ W["fc1"].T + B["fc1"]
    q1 = apply_quantized_multiplier(acc1, *M_const["fc1"])
    q1 = torch.clamp(q1, 0, 127)                      # ReLU (fused into the int8 clamp)

    acc2 = q1 @ W["fc2"].T + B["fc2"]
    q2 = apply_quantized_multiplier(acc2, *M_const["fc2"])
    q2 = torch.clamp(q2, 0, 127)

    acc3 = q2 @ W["fc3"].T + B["fc3"]                  # raw int32 logits, no requant needed
    return acc3


@torch.no_grad()
def evaluate_integer_only(loader):
    correct, total = 0, 0
    for x, y in loader:
        logits = integer_only_forward(x)
        correct += (logits.argmax(1) == y).sum().item()
        total += y.size(0)
    return correct / total


int_only_acc = evaluate_integer_only(test_loader)
print(f"fp32 baseline:                      {ref_fp32_acc*100:.2f}%")
print(f"PTQ int8 (float dequant/requant):   {ref_int8_acc*100:.2f}%")
print(f"PTQ int8 (integer-only M decomp):   {int_only_acc*100:.2f}%")

fp32 baseline:                      89.51%
PTQ int8 (float dequant/requant):   89.43%
PTQ int8 (integer-only M decomp):   89.43%


In [6]:
# ----------------------------------------------------------------------
# Save the requantization constants for the RTL/testbench side (ABACUS-9
# onward) and for ABACUS-8's integer-only NumPy golden model.
# ----------------------------------------------------------------------
np.savez(
    "requant_constants.npz",
    fc1_M0=np.int64(M_const["fc1"][0]), fc1_shift=np.int64(M_const["fc1"][1]),
    fc2_M0=np.int64(M_const["fc2"][0]), fc2_shift=np.int64(M_const["fc2"][1]),
    int_only_test_acc=int_only_acc,
)
print("saved -> requant_constants.npz")

saved -> requant_constants.npz
